In [0]:
df = spark.read.csv('dbfs:/FileStore/Products.csv',header=True)

In [0]:
display(df)

OrderDate,ProductKey,Country,EnglishProductName,SalesAmount,UnitPrice,OrderQuantity,TaxAmt,TotalProductCost
01/01/2016,336,Italy,"Road-650 Black, 62",699.0982,699.0982,1,55.9279,413.1463
01/01/2016,222,United States,"Sport-100 Helmet, Blue",34.99,34.99,1,2.7992,13.0863
01/01/2016,540,Canada,HL Road Tire,32.6,32.6,1,2.608,12.1924
01/01/2016,536,Italy,ML Mountain Tire,29.99,29.99,1,2.3992,11.2163
01/01/2016,490,Germany,"Short-Sleeve Classic Jersey, L",53.99,53.99,1,4.3192,41.5723
01/01/2016,225,Canada,AWC Logo Cap,8.99,8.99,1,0.7192,6.9223
01/01/2016,390,United States,"Road-550-W Yellow, 48",1120.49,1120.49,1,89.6392,713.0798
01/01/2016,479,India,Road Bottle Cage,8.99,8.99,1,0.7192,3.3623
01/01/2016,463,United States,"Half-Finger Gloves, S",24.49,24.49,1,1.9592,9.1593
01/01/2016,477,India,Water Bottle - 30 oz.,4.99,4.99,1,0.3992,1.8663


In [0]:
a=df.select("OrderDate","Country",'EnglishProductName')
a.display()

OrderDate,Country,EnglishProductName
01/01/2016,Italy,"Road-650 Black, 62"
01/01/2016,United States,"Sport-100 Helmet, Blue"
01/01/2016,Canada,HL Road Tire
01/01/2016,Italy,ML Mountain Tire
01/01/2016,Germany,"Short-Sleeve Classic Jersey, L"
01/01/2016,Canada,AWC Logo Cap
01/01/2016,United States,"Road-550-W Yellow, 48"
01/01/2016,India,Road Bottle Cage
01/01/2016,United States,"Half-Finger Gloves, S"
01/01/2016,India,Water Bottle - 30 oz.


In [0]:
from pyspark.sql.functions import col
df1=df.select("OrderDate","Country",'EnglishProductName','SalesAmount')

In [0]:
from pyspark.sql.functions import * 
df1=df1.withColumn("Country",lit('India'))
df1=df1.withColumn("Tax",col('SalesAmount')*0.05)
df1.display()

OrderDate,Country,EnglishProductName,SalesAmount,Tax
01/01/2016,India,"Road-650 Black, 62",699.0982,34.954910000000005
01/01/2016,India,"Sport-100 Helmet, Blue",34.99,1.7495000000000003
01/01/2016,India,HL Road Tire,32.6,1.6300000000000001
01/01/2016,India,ML Mountain Tire,29.99,1.4995
01/01/2016,India,"Short-Sleeve Classic Jersey, L",53.99,2.6995000000000005
01/01/2016,India,AWC Logo Cap,8.99,0.4495
01/01/2016,India,"Road-550-W Yellow, 48",1120.49,56.0245
01/01/2016,India,Road Bottle Cage,8.99,0.4495
01/01/2016,India,"Half-Finger Gloves, S",24.49,1.2245
01/01/2016,India,Water Bottle - 30 oz.,4.99,0.24950000000000003


In [0]:
from pyspark.sql import SparkSession
spark=SparkSession.builder.appName('trial')\
    .master('local')\
    .getOrCreate()

In [0]:
df1=spark.read.format('json').option("inferSchema",True)\
    .option('multiline',False)\
    .load('dbfs:/FileStore/drivers.json')

In [0]:
df1.display()

code,dob,driverId,driverRef,name,nationality,number,url
HAM,1985-01-07,1,hamilton,"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton
HEI,1977-05-10,2,heidfeld,"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld
ROS,1985-06-27,3,rosberg,"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg
ALO,1981-07-29,4,alonso,"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso
KOV,1981-10-19,5,kovalainen,"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen
NAK,1985-01-11,6,nakajima,"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima
BOU,1979-02-28,7,bourdais,"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais
RAI,1979-10-17,8,raikkonen,"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen
KUB,1984-12-07,9,kubica,"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica
GLO,1982-03-18,10,glock,"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock


In [0]:
### schema definition
df1.printSchema()
df1=df1.select(col('code'),col('driverId'),col('dob'))
# change datatype
field_names='''
code STRING,
driverId STRING,
dob STRING
'''
# df1.display()
df1.schema(field_names).display()

#alias - temporary
df1.select(col('code'),col('dob').alias('Name')).display()

#filter
df1.filter((col('code')=='HAM') & (col('driverId').isNull())).display()

#withColumnRenamed - permamnent in dataframe
df1.withColumnRenamed('codee','code').display()

# withColumn
# lit maps every row with the given value
#                name   transformation
df1.withColumn('code',lit('code')).display()
df1.withColumn('code1',col('driverId')).display()

#regex_replace
df1.withColumn('code',regexp_replace(col('code'),'HAM','AMY'))\
    .withColumn('code',regexp_replace(col('code'),'ROS','ASDDFAA')).display()

#casting
df1.withColumn('driverId',col('driverId').cast('int')).display()

#sort and order by
# sort on 1 col
df1.sort(col('driverId').desc()).display()

# Sort on multiple col
df1.sort(['driverId','code'],ascending=[0,0]).display()

#limit
df1.limit(5).display()

#drop
df1.drop('code','driverId').display()

#drop duplicates dedupe
# duplicate rows
df1.dropDuplicates().display()

#duplicate row in a column
df1.dropDuplicates(subset=['driverId','code']).display()

# disitinct
df1.select(col('code'),col('driverId')).distinct().display()


root
 |-- code: string (nullable = true)
 |-- dob: string (nullable = true)
 |-- driverId: long (nullable = true)
 |-- driverRef: string (nullable = true)
 |-- name: struct (nullable = true)
 |    |-- forename: string (nullable = true)
 |    |-- surname: string (nullable = true)
 |-- nationality: string (nullable = true)
 |-- number: string (nullable = true)
 |-- url: string (nullable = true)



In [0]:
d1=[('1','kad'),('2','sid')]
schema1='id STRING, name STRING'

d2=[('3','rahul'),('4','jas')]
schema2='id STRING, name STRING'

df1=spark.createDataFrame(d1,schema=schema1)
df2=spark.createDataFrame(d2,schema=schema2)

#union
df1.union(df2).display()


id,name
1,kad
2,sid
3,rahul
4,jas


In [0]:
# union by name

d1=d1=[('kad','1'),('sid','2')]
schema3='name STRING, id STRING'
df3=spark.createDataFrame(d1,schema=schema3)
# df3.display()
#unionByName matches the column name and does union
df2.unionByName(df3).display()

In [0]:
#string functions
#initcap, 
df1=spark.read.format('json').option("inferSchema",True)\
    .option('multiline',False)\
    .load('dbfs:/FileStore/drivers.json')
df1.select(initcap('driverRef').alias('Firts cap'),lower('driverRef').alias('All lower'),upper('driverRef').alias('All upper')).display()

Firts cap,All lower,All upper
Hamilton,hamilton,HAMILTON
Heidfeld,heidfeld,HEIDFELD
Rosberg,rosberg,ROSBERG
Alonso,alonso,ALONSO
Kovalainen,kovalainen,KOVALAINEN
Nakajima,nakajima,NAKAJIMA
Bourdais,bourdais,BOURDAIS
Raikkonen,raikkonen,RAIKKONEN
Kubica,kubica,KUBICA
Glock,glock,GLOCK


In [0]:
# date functions
#current date
df1=df1.withColumn('curr_Date',current_date())
# df1.display()

# add date
df1= df1.withColumn('week_After',date_add('curr_Date',7))
#date sub
# df1=df1.withColumn('week_before',date_sub('curr_Date',7))
# or
df1=df1.withColumn('week_before',date_add('curr_Date',-7))
# df1.display()

#dateDiff
df1.withColumn('date_diff',datediff('week_before','week_after')).display()
df1.select(datediff('week_after','week_before')).display()

#date_format
df1=df1.withColumn('week_before',date_format('week_before','yyyy'))
df1.select(col('week_before')).display()


week_before
2025
2025
2025
2025
2025
2025
2025
2025
2025
2025


In [0]:
#handling null values
# dropping null 3 methods
    # all= if all row is null than only drop
    # any= if any one column has null then row is drop
    # susbet= if that col is null than drop
    # df1.dropna(how='any').display()
    # df1.dropna(how='all').display()
    # df1.dropna(subset=['code']).display()
# replace null 
df1.fillna('NA',subset=['code','driverID']).display()


code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before
HAM,1985-01-07,1,hamilton,"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025
HEI,1977-05-10,2,heidfeld,"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025
ROS,1985-06-27,3,rosberg,"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025
ALO,1981-07-29,4,alonso,"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025
KOV,1981-10-19,5,kovalainen,"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025
NAK,1985-01-11,6,nakajima,"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025
BOU,1979-02-28,7,bourdais,"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025
RAI,1979-10-17,8,raikkonen,"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025
KUB,1984-12-07,9,kubica,"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025
GLO,1982-03-18,10,glock,"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025


In [0]:
#split and indexing(idex for accessing splitted values)
df1.withColumn('driverRef',split('driverRef','_')[1]).display()



code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before
HAM,1985-01-07,1,null,"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025
HEI,1977-05-10,2,null,"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025
ROS,1985-06-27,3,null,"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025
ALO,1981-07-29,4,null,"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025
KOV,1981-10-19,5,null,"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025
NAK,1985-01-11,6,null,"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025
BOU,1979-02-28,7,null,"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025
RAI,1979-10-17,8,null,"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025
KUB,1984-12-07,9,null,"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025
GLO,1982-03-18,10,null,"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025


In [0]:
#explode function - if cell has multiple values it will segregate into different rows creating multiple rows
df2=df1.withColumn('driverRef',split('driverRef','_'))
df2.withColumn('driverRef',explode('driverRef')).display()

code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before
HAM,1985-01-07,1,hamilton,"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025
HEI,1977-05-10,2,heidfeld,"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025
ROS,1985-06-27,3,rosberg,"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025
ALO,1981-07-29,4,alonso,"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025
KOV,1981-10-19,5,kovalainen,"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025
NAK,1985-01-11,6,nakajima,"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025
BOU,1979-02-28,7,bourdais,"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025
RAI,1979-10-17,8,raikkonen,"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025
KUB,1984-12-07,9,kubica,"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025
GLO,1982-03-18,10,glock,"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025


In [0]:
#just creating sample data with col as list
df3=df1.withColumn('driverRef',split('driverRef','_'))
# df3.display()

# array_contains
df3.withColumn('schumacher_surname',array_contains('driverRef','schumacher')).display()

code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before,schumacher_surname
HAM,1985-01-07,1,List(hamilton),"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025,false
HEI,1977-05-10,2,List(heidfeld),"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025,false
ROS,1985-06-27,3,List(rosberg),"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025,false
ALO,1981-07-29,4,List(alonso),"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025,false
KOV,1981-10-19,5,List(kovalainen),"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025,false
NAK,1985-01-11,6,List(nakajima),"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025,false
BOU,1979-02-28,7,List(bourdais),"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025,false
RAI,1979-10-17,8,List(raikkonen),"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025,false
KUB,1984-12-07,9,List(kubica),"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025,false
GLO,1982-03-18,10,List(glock),"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025,false


In [0]:
#group by
df3.groupBy('nationality','code').agg(sum('driverId'),avg('driverId')).display()

nationality,code,sum(driverId),avg(driverId)
Dutch,ALB,27,27.0
Malaysian,\N,62,62.0
New Zealander,HAR,843,843.0
Thai,ALB,848,848.0
Finnish,RAI,8,8.0
Argentine-Italian,\N,573,573.0
British,COU,14,14.0
Russian,PET,808,808.0
Russian,MAZ,853,853.0
Venezuelan,\N,691,345.5


In [0]:
#################### Advance ####################

#collect_list
df3.filter(col('code')!="\\N").groupBy('nationality').agg(collect_list('code')).display()
# df3.groupBy('nationality').agg(collect_list('code')).display()

nationality,collect_list(code)
Mexican,"List(PER, GUT)"
Finnish,"List(KOV, RAI, BOT)"
Swiss,List(BUE)
Thai,List(ALB)
Indian,"List(KAR, CHA)"
Indonesian,List(HAR)
Polish,List(KUB)
British,"List(HAM, COU, BUT, DAV, DIR, CHI, STE, PAL, NOR, RUS, AIT)"
Japanese,"List(NAK, SAT, YAM, IDE, KOB, TSU)"
Australian,"List(WEB, RIC)"


In [0]:
#pivot

df3.groupBy('code').pivot('nationality').agg(count('driverId')).display()


code,American,American-Italian,Argentine,Argentine-Italian,Australian,Austrian,Belgian,Brazilian,British,Canadian,Chilean,Colombian,Czech,Danish,Dutch,East German,Finnish,French,German,Hungarian,Indian,Indonesian,Irish,Italian,Japanese,Liechtensteiner,Malaysian,Mexican,Monegasque,New Zealander,Polish,Portuguese,Rhodesian,Russian,South African,Spanish,Swedish,Swiss,Thai,Uruguayan,Venezuelan
ZON,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
OCO,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
STE,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
NAS,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
DIR,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
ALG,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1,null,null,null,null,null
BIA,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
DAM,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null
MAL,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,1
DIG,null,null,null,null,null,null,null,1,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null,null


In [0]:
#when otherwise
# df3.withColumn('Asian?',when(col('nationality')=='German','European').otherwise('NA')).display()
# Scenario1
df3.withColumn('lotarey',when(((col('nationality')=='British') & (col('driverId')>=10)),\
                         when(col('code')=='HAM','Yes')).otherwise('NA')).display()



code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before,lotarey
HAM,1985-01-07,1,List(hamilton),"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025,NA
HEI,1977-05-10,2,List(heidfeld),"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025,NA
ROS,1985-06-27,3,List(rosberg),"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025,NA
ALO,1981-07-29,4,List(alonso),"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025,NA
KOV,1981-10-19,5,List(kovalainen),"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025,NA
NAK,1985-01-11,6,List(nakajima),"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025,NA
BOU,1979-02-28,7,List(bourdais),"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025,NA
RAI,1979-10-17,8,List(raikkonen),"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025,NA
KUB,1984-12-07,9,List(kubica),"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025,NA
GLO,1982-03-18,10,List(glock),"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025,NA


In [0]:
#joins
#inner join
df1=df3.select(['code','driverId'])
df2=df3.select(['nationality','driverId'])
df1.join(df2,on=df1['driverId']==df2['driverId'],how='inner').display()

code,driverId,nationality,driverId
HAM,1,British,1
HEI,2,German,2
ROS,3,German,3
ALO,4,Spanish,4
KOV,5,Finnish,5
NAK,6,Japanese,6
BOU,7,French,7
RAI,8,Finnish,8
KUB,9,Polish,9
GLO,10,German,10


In [0]:
#left join and right join
df1.join(df2,on=df1['driverId']==df2['driverId'],how='left').display()


code,driverId,nationality,driverId
HAM,1,British,1
HEI,2,German,2
ROS,3,German,3
ALO,4,Spanish,4
KOV,5,Finnish,5
NAK,6,Japanese,6
BOU,7,French,7
RAI,8,Finnish,8
KUB,9,Polish,9
GLO,10,German,10


In [0]:
#anti join - fetch row from df1 which are not there in df2
df1.join(df2,on=df1['driverId']==df2['driverId'],how='anti').display()

code,driverId


In [0]:
# window functions
# 1)row_number - unique number for every row
from pyspark.sql.window import Window

# df3.withColumn('row_num',row_number().over(Window.partitionBy('nationality').orderBy(col('nationality').asc())))\
# .select(['nationality','row_num']).display()

# 2)rank()
# df3.withColumn('rank',rank().over(Window.orderBy(col('nationality').desc()))).select(['nationality','rank']).display()

# 3)dense_rank
# df3.withColumn('rank',rank().over(Window.orderBy(col('nationality')))).\
    # withColumn('dense_rank',dense_rank().over(Window.orderBy('nationality'))).select(['nationality','rank','dense_rank']).display()

# 4)cumsum
df3.withColumn('CumSum',sum('driverId').over(Window.orderBy('nationality').rowsBetween(Window.unboundedPreceding,Window.currentRow)))\
    .select(col('nationality'),col('CumSum'),col('driverId')).display()


nationality,CumSum,driverId
American,26,26
American,147,121
American,305,158
American,498,193
American,705,207
American,919,214
American,1158,239
American,1398,240
American,1647,249
American,1935,288


In [0]:
# user defined functions (udf). Do not use it as it is not well optimized.
def sqr(x):
    return x**2
#creating udf function
my_udf=udf(sqr) # this is now in pyspark
df3.withColumn('square',my_udf(col('driverId'))).display()




code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before,square
HAM,1985-01-07,1,List(hamilton),"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025,1
HEI,1977-05-10,2,List(heidfeld),"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025,4
ROS,1985-06-27,3,List(rosberg),"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025,9
ALO,1981-07-29,4,List(alonso),"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025,16
KOV,1981-10-19,5,List(kovalainen),"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025,25
NAK,1985-01-11,6,List(nakajima),"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025,36
BOU,1979-02-28,7,List(bourdais),"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025,49
RAI,1979-10-17,8,List(raikkonen),"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025,64
KUB,1984-12-07,9,List(kubica),"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025,81
GLO,1982-03-18,10,List(glock),"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025,100


In [0]:
#writing into file

# df3.select(col('code'),col('dob'),col('driverId')).write.format('csv')\
    # .save('dbfs:/FileStore/sample.csv')

# data writing modes
# append -add the files
# overwrite
# error -if file exists then through error
# ignore - if file exist don't overwrite and don't through error

df3.select(col('code'),col('dob'),col('driverId'))\
    .write.format('csv').mode('append').save('dbfs:/FileStore/sample.csv')

    #overwrite
df3.select(col('code'),col('dob'),col('driverId'))\
    .write.format('csv').mode('overwrite').save('dbfs:/FileStore/sample.csv')

#ignore
df3.select(col('code'),col('dob'),col('driverId'))\
    .write.format('csv').mode('ignore').save('dbfs:/FileStore/sample.csv')

#error - will through error
# df3.select(col('code'),col('dob'),col('driverId'))\
#     .write.format('csv').mode('error').save('dbfs:/FileStore/sample.csv')




In [0]:
# storing in parquet- headers are stored at the footer of the file
# df3.select(col('code'),col('dob'),col('driverId'))\
    # .write.format('parquet').mode('append').save('dbfs:/FileStore/sample.parquet')

# delta lake format - parquet file + headers is stored in a delta log file

#saveAsTable so that we can sql
df3.select(col('code'),col('dob'),col('driverId'))\
    .write.format('csv').mode('append').saveAsTable('table1')



In [0]:
# #managed vs external tabl
# managed  - by databricks. when we delete the table we delete the data plus metadata
# external -  managed by us. deleting only delete matadata and not data

In [0]:
#spark sql
df3.createTempView('my_view')



  File <command-4185416559317451>:5
    select * from my_view
             ^
SyntaxError: invalid syntax


In [0]:
%sql

select * from my_view


code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before
HAM,1985-01-07,1,List(hamilton),"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025
HEI,1977-05-10,2,List(heidfeld),"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025
ROS,1985-06-27,3,List(rosberg),"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025
ALO,1981-07-29,4,List(alonso),"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025
KOV,1981-10-19,5,List(kovalainen),"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025
NAK,1985-01-11,6,List(nakajima),"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025
BOU,1979-02-28,7,List(bourdais),"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025
RAI,1979-10-17,8,List(raikkonen),"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025
KUB,1984-12-07,9,List(kubica),"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025
GLO,1982-03-18,10,List(glock),"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025


In [0]:
#converting sql result into dataframe
df_sql=spark.sql('select * from my_view')
df_sql.display()

code,dob,driverId,driverRef,name,nationality,number,url,curr_Date,week_After,week_before
HAM,1985-01-07,1,List(hamilton),"List(Lewis, Hamilton)",British,44,http://en.wikipedia.org/wiki/Lewis_Hamilton,2025-04-08,2025-04-15,2025
HEI,1977-05-10,2,List(heidfeld),"List(Nick, Heidfeld)",German,\N,http://en.wikipedia.org/wiki/Nick_Heidfeld,2025-04-08,2025-04-15,2025
ROS,1985-06-27,3,List(rosberg),"List(Nico, Rosberg)",German,6,http://en.wikipedia.org/wiki/Nico_Rosberg,2025-04-08,2025-04-15,2025
ALO,1981-07-29,4,List(alonso),"List(Fernando, Alonso)",Spanish,14,http://en.wikipedia.org/wiki/Fernando_Alonso,2025-04-08,2025-04-15,2025
KOV,1981-10-19,5,List(kovalainen),"List(Heikki, Kovalainen)",Finnish,\N,http://en.wikipedia.org/wiki/Heikki_Kovalainen,2025-04-08,2025-04-15,2025
NAK,1985-01-11,6,List(nakajima),"List(Kazuki, Nakajima)",Japanese,\N,http://en.wikipedia.org/wiki/Kazuki_Nakajima,2025-04-08,2025-04-15,2025
BOU,1979-02-28,7,List(bourdais),"List(Sébastien, Bourdais)",French,\N,http://en.wikipedia.org/wiki/S%C3%A9bastien_Bourdais,2025-04-08,2025-04-15,2025
RAI,1979-10-17,8,List(raikkonen),"List(Kimi, Räikkönen)",Finnish,7,http://en.wikipedia.org/wiki/Kimi_R%C3%A4ikk%C3%B6nen,2025-04-08,2025-04-15,2025
KUB,1984-12-07,9,List(kubica),"List(Robert, Kubica)",Polish,88,http://en.wikipedia.org/wiki/Robert_Kubica,2025-04-08,2025-04-15,2025
GLO,1982-03-18,10,List(glock),"List(Timo, Glock)",German,\N,http://en.wikipedia.org/wiki/Timo_Glock,2025-04-08,2025-04-15,2025
